<a href="https://colab.research.google.com/github/ipeirotis-org/datasets/blob/main/MTurk_Tracker/MTurk_Tracker_Tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The MTurk Tracker Dataset — A Researcher's Guide

[MTurk Tracker](https://www.mturk-tracker.com) longitudinally crawled the **public Amazon Mechanical Turk marketplace**, recording every batch of tasks that appeared and how it evolved over time. This notebook is both **documentation** and a **hands-on tutorial** for the copy of that data now hosted in BigQuery at **`nyu-datasets.mturk_tracker`**.

## What this dataset is
- A repeated **crawl of the "HITs available" listings** on MTurk. The unit is the **HIT group** (a *batch*: a set of identical Human Intelligence Tasks posted together by one requester). Each batch is re-observed on every crawl, giving a **time series of how many HITs were still available**.
- It is a **market-/requester-side** dataset: task listings, rewards, requesters, and availability over time. **It contains no worker-level or personally identifiable information.**

## Coverage & scale (this BigQuery copy)
| | |
|---|---|
| **Time span** | `2014-05-01` → `2015-02-18` (UTC) |
| **HIT groups (batches)** | 433,156 |
| **Distinct requesters** | 13,454 |
| **Crawl observations** (`hit_instances`) | 16,260,725 |
| **Task HTML records** (`hit_content`) | 423,824 (338,021 with HTML) |
| **Market snapshots** (`market_statistics`) | 409,622 |
| **Hourly flow rows** (`arrival_completions`) | 7,026 |
| **Requester ranking snapshots** (`top_requesters`) | 566,468 |

> This is the data preserved in the tracker's datastore for this window. The marketplace totals quoted in the paper below (millions of batches over many years) span the tracker's full lifetime; this BigQuery copy is the ~9.5-month slice that survives in the datastore export.

## Origin & citation
Collected by the **MTurk Tracker** (Panagiotis G. Ipeirotis and collaborators). The methodology and a large-scale analysis appear in:

> Djellel Eddine Difallah, Michele Catasta, Gianluca Demartini, Panagiotis G. Ipeirotis, Philippe Cudré-Mauroux. **"The Dynamics of Micro-Task Crowdsourcing: The Case of Amazon MTurk."** *WWW 2015.*

If you use this data, please cite that paper (BibTeX at the end of this notebook).

## What it is good for
Arrival / throughput processes, **queueing and matching models**, dynamic pricing, market-size and seasonality trends, requester behavior, and task taxonomy (from titles/keywords).

## Data model

```
hit_requesters (1) ──< hit_groups (1) ──< hit_instances (many)      time series of availability
                          │
                          └──(1)──────────  hit_content             rendered task HTML

market_statistics     market-wide totals (HITs / groups available) at each crawl
arrival_completions   market-wide HOURLY flow: HITs/groups/rewards arrived & completed
top_requesters        periodic snapshots of the largest requesters
batch_observations    VIEW = hit_instances ⨝ hit_groups ⨝ hit_requesters  (the modeling panel)
```

**The join keys**
- `hit_instances.group_id` → `hit_groups.group_id` (a batch's observations).
- `hit_groups.requester_id` → `hit_requesters.requester_id` (who posted it).
- `hit_content.group_id` → `hit_groups.group_id` (the task's HTML).

Every clean table also has a lossless **`*_raw`** companion holding the original Datastore entity JSON as a single `STRING` column — use it only if you need a field not surfaced in the typed tables.

## 1. Setup and authentication

The dataset lives in the `nyu-datasets` project. To query it you need:
1. A Google account that has been granted **BigQuery Data Viewer** on `nyu-datasets.mturk_tracker` (ask the dataset owner if you don't have it), and
2. **Your own** Google Cloud project — BigQuery bills the bytes each query scans to *your* project, not to `nyu-datasets`.

In [ ]:
# In Google Colab, google-cloud-bigquery, pandas and matplotlib are pre-installed.
# Locally, install them first:
#   pip install google-cloud-bigquery db-dtypes pandas matplotlib
from google.cloud import bigquery
from google.colab import auth

auth.authenticate_user()   # sign in with the Google account that has access

# >>> CHANGE THIS to a Google Cloud project where YOU can run BigQuery jobs <<<
PROJECT_ID = "your-gcp-project"

client = bigquery.Client(project=PROJECT_ID)

import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 120)

def q(sql: str) -> pd.DataFrame:
    """Run a SQL string on BigQuery and return a pandas DataFrame."""
    return client.query(sql).to_dataframe()

print("Connected. Querying nyu-datasets.mturk_tracker as project:", PROJECT_ID)

## 2. What's in the dataset (inventory)

Row counts and on-disk sizes of every table/view. The `*_raw` tables are the lossless staging copies; the lowercase typed tables and the `batch_observations` view are what you'll normally use.

In [ ]:
q("""
SELECT table_id AS name,
       row_count,
       ROUND(size_bytes/1e9, 3) AS size_gb,
       CASE type WHEN 1 THEN 'table' WHEN 2 THEN 'view' ELSE CAST(type AS STRING) END AS kind
FROM `nyu-datasets.mturk_tracker.__TABLES__`
ORDER BY row_count DESC
""")

## 3. Data dictionary

### `hit_groups` — one row per batch (HIT group)
| column | type | description |
|---|---|---|
| `group_id` | STRING | Batch id (the HIT group id on MTurk). Primary key. |
| `requester_id` | STRING | Public MTurk requester id; join to `hit_requesters`. |
| `title` | STRING | HIT title shown to workers. |
| `description` | STRING | HIT short description. |
| `keywords` | ARRAY&lt;STRING&gt; | Requester-supplied keywords (closest thing to a task *type*). |
| `reward_cents` | INT64 | Reward per HIT, in **US cents**. (Rarely `-100` = a sentinel for "not captured".) |
| `reward_usd` | FLOAT64 | `reward_cents / 100`. |
| `time_alloted_seconds` | INT64 | Time allotted per assignment, in **seconds**. |
| `first_seen` | TIMESTAMP | First time the crawler saw this batch (UTC). A good proxy for the batch's **arrival/start time**. |
| `last_seen` | TIMESTAMP | Last time the batch was seen available (UTC). |
| `expiration_date` | TIMESTAMP | Requester-set expiration (UTC). |
| `active` | BOOL | Whether the batch was still active at the last crawl. |

### `hit_instances` — the availability time series (16.3M rows)
| column | type | description |
|---|---|---|
| `group_id` | STRING | Batch this observation belongs to; join to `hit_groups`. |
| `observation_time` | TIMESTAMP | Crawl time of this observation (UTC). |
| `hits_available` | INT64 | HITs still available in the batch at this moment. |
| `rewards_available` | INT64 | Total reward available in the batch = `hits_available × reward_cents` (cents). |
| `hits_diff` | INT64 | Change in `hits_available` since the previous observation (**+** = HITs added, **−** = HITs taken/expired). |
| `reward_diff` | INT64 | Change in `rewards_available` since the previous observation (cents). |

### `hit_requesters` — requester directory
| column | type | description |
|---|---|---|
| `requester_id` | STRING | Public MTurk requester id (primary key). |
| `requester_name` | STRING | Display name. |
| `last_activity` | TIMESTAMP | Last time this requester was seen active (UTC). |

### `top_requesters` — periodic "largest requesters" snapshots
| column | type | description |
|---|---|---|
| `requester_id`, `requester_name` | STRING | The requester. |
| `snapshot_time` | TIMESTAMP | When this ranking snapshot was taken (UTC). |
| `hits` | INT64 | Cumulative HITs attributed to the requester at the snapshot. |
| `reward_cents` | INT64 | Cumulative reward (cents). |
| `projects` | INT64 | Project count (often 0 in this period). |

### `market_statistics` — market-wide availability snapshots
| column | type | description |
|---|---|---|
| `timestamp` | TIMESTAMP | Crawl time (UTC). |
| `hits_available` | INT64 | Total HITs available across the whole marketplace. |
| `hit_groups_available` | INT64 | Total distinct batches available across the marketplace. |

### `arrival_completions` — market-wide HOURLY flow (the authoritative arrival/throughput series)
| column | type | description |
|---|---|---|
| `from_time`, `to_time` | TIMESTAMP | The hour covered (UTC). |
| `length_minutes` | INT64 | Window length (≈60). |
| `hits_arrived`, `hits_completed` | INT64 | HITs that appeared / were completed during the hour. |
| `hit_groups_arrived`, `hit_groups_completed` | INT64 | Batches that appeared / finished during the hour. |
| `rewards_arrived`, `rewards_completed` | INT64 | Reward (cents) that arrived / was completed. |
| `hits_available_ui`, `hit_groups_available_ui` | FLOAT64 | Time-averaged availability during the hour. |

### `hit_content` — the rendered task
| column | type | description |
|---|---|---|
| `group_id` | STRING | Batch id; join to `hit_groups`. |
| `content` | STRING | The HIT's HTML (NULL for ~20% of batches). |
| `has_content` | BOOL | Convenience flag: `content IS NOT NULL`. |

### `batch_observations` — VIEW (the modeling panel)
`hit_instances` joined to its `hit_groups` attributes and `hit_requesters` name, plus two **derived** flow columns: `hits_arrived = GREATEST(hits_diff, 0)` and `hits_completed = GREATEST(-hits_diff, 0)`. See the caveats section before relying on these.

In [ ]:
# The live schema, straight from BigQuery, for cross-checking the dictionary above.
q("""
SELECT table_name, column_name, data_type
FROM `nyu-datasets.mturk_tracker.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name IN ('hit_groups','hit_instances','hit_requesters','top_requesters',
                     'market_statistics','arrival_completions','hit_content','batch_observations')
ORDER BY table_name, ordinal_position
""")

## 4. `hit_groups` — the batches

One row per batch. Start here to understand *what* was posted: rewards, titles, keywords, timing.

In [ ]:
q("""
SELECT group_id, requester_id, title, reward_usd, time_alloted_seconds,
       ARRAY_TO_STRING(keywords, ', ') AS keywords, first_seen, active
FROM `nyu-datasets.mturk_tracker.hit_groups`
ORDER BY first_seen
LIMIT 8
""")

In [ ]:
# Reward distribution (the classic MTurk "most tasks are cheap" shape).
rw = q("""
SELECT ROUND(reward_usd, 2) AS reward_usd, COUNT(*) AS n_batches
FROM `nyu-datasets.mturk_tracker.hit_groups`
WHERE reward_cents BETWEEN 0 AND 300      -- $0–$3; drops the rare -100 sentinel and the long tail
GROUP BY reward_usd
ORDER BY reward_usd
""")
plt.figure(figsize=(10,4))
plt.bar(rw.reward_usd, rw.n_batches, width=0.04)
plt.xlabel("Reward per HIT (USD)"); plt.ylabel("# batches")
plt.title("Reward distribution across HIT groups (capped at $3)")
plt.show()
print("Median reward (USD):",
      q("SELECT ROUND(APPROX_QUANTILES(reward_usd,2)[OFFSET(1)],2) m "
        "FROM `nyu-datasets.mturk_tracker.hit_groups` WHERE reward_cents>=0").m.iloc[0])

In [ ]:
# Most common task keywords (a rough task taxonomy).
q("""
SELECT kw, COUNT(*) AS n_batches
FROM `nyu-datasets.mturk_tracker.hit_groups`, UNNEST(keywords) AS kw
WHERE kw != ''
GROUP BY kw
ORDER BY n_batches DESC
LIMIT 25
""")

## 5. `hit_instances` — the availability time series

This is the heart of the dataset: each row says *"at this moment, this batch had N HITs available."* Differencing it (already done for you as `hits_diff`) gives the inflow/outflow of work. Let's visualize one busy batch.

In [ ]:
# Find long-lived, high-volume batches worth plotting.
busy = q("""
SELECT group_id, COUNT(*) AS observations, MAX(hits_available) AS peak_available
FROM `nyu-datasets.mturk_tracker.hit_instances`
GROUP BY group_id
HAVING observations > 300
ORDER BY peak_available DESC
LIMIT 5
""")
busy

In [ ]:
# Availability over time for the single busiest batch above.
gid = busy.group_id.iloc[0]
ts = q(f"""
SELECT observation_time, hits_available
FROM `nyu-datasets.mturk_tracker.hit_instances`
WHERE group_id = '{gid}'
ORDER BY observation_time
""")
plt.figure(figsize=(11,4))
plt.plot(ts.observation_time, ts.hits_available, linewidth=0.8)
plt.title(f"HITs available over time — batch {gid}")
plt.ylabel("HITs available"); plt.xlabel("Observation time (UTC)")
plt.show()

## 6. `market_statistics` — the whole marketplace over time

Market-wide totals at each crawl: how big the "available work" pool was, day by day.

In [ ]:
ms = q("""
SELECT DATE(timestamp) AS day,
       AVG(hits_available)        AS avg_hits_available,
       AVG(hit_groups_available)  AS avg_groups_available
FROM `nyu-datasets.mturk_tracker.market_statistics`
GROUP BY day ORDER BY day
""")
fig, ax1 = plt.subplots(figsize=(11,4))
ax1.plot(ms.day, ms.avg_hits_available, color="tab:blue", label="HITs available")
ax1.set_ylabel("Avg HITs available", color="tab:blue"); ax1.set_xlabel("Day")
ax2 = ax1.twinx()
ax2.plot(ms.day, ms.avg_groups_available, color="tab:orange", label="Batches available")
ax2.set_ylabel("Avg batches available", color="tab:orange")
plt.title("Marketplace size over time")
plt.show()

## 7. `arrival_completions` — hourly arrivals vs. completions

The **authoritative** market-wide flow (don't reconstruct it from diffs if this serves your need). Here is the average intraday pattern of HITs arriving and being completed, by hour of day (UTC).

In [ ]:
ac = q("""
SELECT EXTRACT(HOUR FROM from_time) AS hour_utc,
       AVG(hits_arrived)   AS avg_hits_arrived,
       AVG(hits_completed) AS avg_hits_completed
FROM `nyu-datasets.mturk_tracker.arrival_completions`
GROUP BY hour_utc ORDER BY hour_utc
""")
plt.figure(figsize=(10,4))
plt.plot(ac.hour_utc, ac.avg_hits_arrived,   marker="o", label="arrived")
plt.plot(ac.hour_utc, ac.avg_hits_completed, marker="o", label="completed")
plt.xlabel("Hour of day (UTC)"); plt.ylabel("Avg HITs / hour")
plt.title("Average hourly HIT arrivals vs. completions"); plt.legend()
plt.show()

## 8. Requesters: `top_requesters` and `hit_requesters`

Who supplies the work. `top_requesters` holds periodic ranking snapshots; here are the largest requesters by peak cumulative HITs.

In [ ]:
q("""
SELECT requester_name, MAX(hits) AS peak_cumulative_hits
FROM `nyu-datasets.mturk_tracker.top_requesters`
WHERE requester_name IS NOT NULL
GROUP BY requester_name
ORDER BY peak_cumulative_hits DESC
LIMIT 15
""")

## 9. `hit_content` — the task HTML

The rendered HTML that workers saw, keyed by `group_id` (NULL for ~20% of batches). Useful for NLP / task-type classification beyond the keywords.

In [ ]:
ex = q("""
SELECT group_id, content
FROM `nyu-datasets.mturk_tracker.hit_content`
WHERE has_content
LIMIT 1
""")
print("group_id:", ex.group_id.iloc[0])
print(ex.content.iloc[0][:1500], "...")

## 10. `batch_observations` — the modeling panel

A convenience VIEW that joins each availability observation to its batch and requester attributes, with `hits_arrived` / `hits_completed` derived per-batch from `hits_diff`. This is usually where matching / queueing studies start.

In [ ]:
q("""
SELECT observation_time, group_id, requester_id, requester_name,
       reward_usd, time_alloted_seconds,
       hits_available, hits_arrived, hits_completed
FROM `nyu-datasets.mturk_tracker.batch_observations`
WHERE reward_usd >= 0
ORDER BY observation_time
LIMIT 10
""")

## 11. Worked example — a market arrival process for a queueing / matching model

A common starting point: treat each **arriving HIT** (or batch) as a job with a *reward* (price) and a *time-allotted* (service spec), arriving over time. Below we build the hourly arrival stream and overlay batch arrivals, then pair it with the reward ("type") distribution.

In [ ]:
# Hourly market arrival stream (one full week for readability).
arr = q("""
SELECT TIMESTAMP_TRUNC(from_time, HOUR) AS hour,
       SUM(hits_arrived)        AS hits_arrived,
       SUM(hit_groups_arrived)  AS batches_arrived
FROM `nyu-datasets.mturk_tracker.arrival_completions`
WHERE from_time BETWEEN TIMESTAMP('2014-06-02') AND TIMESTAMP('2014-06-09')
GROUP BY hour ORDER BY hour
""")
fig, ax1 = plt.subplots(figsize=(11,4))
ax1.plot(arr.hour, arr.hits_arrived, color="tab:blue", label="HITs arrived")
ax1.set_ylabel("HITs arrived / hour", color="tab:blue")
ax2 = ax1.twinx()
ax2.plot(arr.hour, arr.batches_arrived, color="tab:green", alpha=0.6, label="Batches arrived")
ax2.set_ylabel("Batches arrived / hour", color="tab:green")
plt.title("Market arrival stream — one week (Jun 2014, UTC)")
plt.show()

# The "type" of each arriving job: reward distribution of batches arriving that week.
q("""
SELECT
  CASE
    WHEN reward_usd = 0           THEN '0 (free)'
    WHEN reward_usd <= 0.05       THEN '(0, 0.05]'
    WHEN reward_usd <= 0.25       THEN '(0.05, 0.25]'
    WHEN reward_usd <= 1.00       THEN '(0.25, 1.00]'
    ELSE '> 1.00' END AS reward_bucket,
  COUNT(*) AS n_batches
FROM `nyu-datasets.mturk_tracker.hit_groups`
WHERE reward_cents >= 0
GROUP BY reward_bucket
ORDER BY MIN(reward_cents)
""")

## 12. Caveats & gotchas (read before modeling)

- **Time zone:** every timestamp is **UTC**. Convert if you want US-Eastern "business hours".
- **Reward units:** `reward_cents` is **integer US cents**; use `reward_usd` for dollars. `reward_cents = -100` is a rare **sentinel** for "reward not captured" (filter `reward_cents >= 0`). A reward of `0` is genuine (free/volunteer HITs exist).
- **Derived flows:** in `batch_observations`, `hits_arrived` / `hits_completed` are derived from the **sign of `hits_diff`** (net availability change). A *decrease* can mean a HIT was **completed _or_ expired/cancelled**, so per-batch completions are an approximation. For authoritative market-wide arrivals/completions, use **`arrival_completions`**.
- **Crawl cadence is not perfectly uniform.** Observations are frequent but irregular; there are gaps. Don't assume a fixed sampling interval — use `observation_time` explicitly and, if you need an evenly spaced series, resample.
- **Not captured by the crawler:** worker **approval rate**, worker **location** qualifications, and the **"Masters"** qualification are **not** in this data. A formal `task_type` field does not exist — derive it from `keywords` / `title` / `content`.
- **`rewards_available = hits_available × reward_cents`** (in cents) — it is *available* reward, not paid reward.
- **`*_raw` tables** keep the original Datastore entity JSON (as a STRING) if you need a field not surfaced here, e.g. `JSON_VALUE(raw, '$.properties.<field>.<valueType>')`.
- **No PII:** this is marketplace/requester-side data only; there is no worker information.

## 13. Access, citation & license

**Access.** The data is in BigQuery at `nyu-datasets.mturk_tracker`. If your `auth.authenticate_user()` account cannot query it, ask the dataset owner (Panos Ipeirotis) to grant your Google account **BigQuery Data Viewer** on the dataset. You query it from, and pay for scanned bytes with, **your own** Google Cloud project.

**Citation.** Please cite:

```bibtex
@inproceedings{difallah2015dynamics,
  title     = {The Dynamics of Micro-Task Crowdsourcing: The Case of Amazon MTurk},
  author    = {Difallah, Djellel Eddine and Catasta, Michele and Demartini, Gianluca
               and Ipeirotis, Panagiotis G. and Cudr{\'e}-Mauroux, Philippe},
  booktitle = {Proceedings of the 24th International Conference on World Wide Web (WWW)},
  year      = {2015}
}
```

**Background.** MTurk Tracker: <https://www.mturk-tracker.com> · Project notebooks: <https://github.com/ipeirotis-org/datasets>

*Questions or access requests: open an issue on the [datasets repo](https://github.com/ipeirotis-org/datasets) or contact the maintainer.*